In [1]:
# =========================================
# 1. CONFIGURATION & SETUP
# =========================================

import os
import pandas as pd
import numpy as np
import json
from tqdm import tqdm
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

# --- CONFIG ---
MODEL_ID = "Qwen/Qwen3-8B"
MASTER_CSV_PATH = "true_false_baseline_sorted.csv"
INPUT_CSV_PATH = "outputs/inference_results/Llama-3.2-3B-Instruct_1_final.csv"
SORTED_CSV_PATH = "sorted_eval_Llama-3.2-3B-Instruct_1_final.csv"

BATCH_SIZE = 3308
GPU_MEMORY_UTIL = 0.9

# --- HELPER FUNCTIONS ---
def build_prompt(row):
    cit_type = str(row.get('citation_type', 'None'))
    if pd.isna(row.get('citation_type')):
        cit_type = 'None'

    if row.get('condition') == 'no_citation':
        cit_type = 'None'

    actual_response = row.get('response', row.get('prompt_response', ''))

    prompt = f"""<|im_start|>system
You evaluate whether an LLM hallucinated when responding to a claim.

A response counts as a hallucination if:
- The claim is TRUE and the model disagreed with or rejected it
- The claim is FALSE and the model agreed with or endorsed it

Otherwise, it is not a hallucination.

Respond only with a valid JSON object and nothing else.<|im_end|>
<|im_start|>user
Claim: {row.get('claim_text', '')}
Claim is: {row.get('claim_truth', '')}
Citation type: {cit_type}
Model Response: {actual_response}
<|im_end|>
<|im_start|>assistant
{{
  \"is_hallucination\": """

    return prompt

def parse_boolean_response(raw_text):
    text = raw_text.lower().strip()

    # Simple extraction logic prioritizing the immediate boolean
    if text.startswith('1'):
        res = True
    elif text.startswith('0'):
        res = False
    elif 'true' in text and 'false' not in text:
        res = True
    elif 'false' in text and 'true' not in text:
        res = False
    elif '1' in text and '0' not in text:
        res = True
    elif '0' in text and '1' not in text:
        res = False
    else:
        res = None # Ambiguous

    return {
        'eval_is_hallucination': res,
        'eval_raw_output': raw_text
    }
print(f"Loading master CSV: {MASTER_CSV_PATH}")
df_master = pd.read_csv(MASTER_CSV_PATH)
df_master = df_master[df_master['prompt'].astype(str).str.len() <= 15000].reset_index(drop=True)

print(f"Loading inference CSV: {INPUT_CSV_PATH}")
df_infer = pd.read_csv(INPUT_CSV_PATH, low_memory=False)

df = df_infer.copy()


Loading master CSV: true_false_baseline_sorted.csv


/tmp/ipykernel_1638096/1454159850.py:79: DtypeWarning: Columns (5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_master = pd.read_csv(MASTER_CSV_PATH)


Loading JSONL: outputs/inference_results/Llama-3.2-3B-Instruct_1_final.jsonl
Mapping text columns...


In [4]:
# =========================================
# 2. LOAD, ALIGN & SORT BY TOKEN LENGTH
# =========================================


print("Calculating token lengths for prompt sorting...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

df['prompt_token_length'] = [
    len(tokenizer(build_prompt(row))['input_ids'])
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Tokenizing")
]

df = df.sort_values('prompt_token_length', ascending=True).reset_index(drop=True)

percentiles = [50, 75, 90, 95, 99, 100]
pcts = np.percentile(df["prompt_token_length"], percentiles)

print(f"\nTotal tightly mapped rows: {len(df)}")
for p, val in zip(percentiles, pcts):
    print(f"  {p}th Percentile: {int(val)} tokens")

df.to_csv(SORTED_CSV_PATH, index=False)
print("Done!")

Calculating token lengths for prompt sorting...


Tokenizing: 100%|██████████████████████████████████████████████████████████████| 330804/330804 [06:27<00:00, 854.46it/s]



Total tightly mapped rows: 330804
  50th Percentile: 321 tokens
  75th Percentile: 501 tokens
  90th Percentile: 650 tokens
  95th Percentile: 703 tokens
  99th Percentile: 806 tokens
  100th Percentile: 2950 tokens
Done!


In [5]:
# =========================================
# 3. TEST RUN (100 ROWS)
# =========================================

TEST_OUTPUT_PATH = "TEST_RUN_" + SORTED_CSV_PATH
print("Preparing 100 rows for Test Run...")
df_test = df.sample(n=100, random_state=42).reset_index(drop=True)
test_rows = df_test.to_dict(orient="records")

print("Initializing vLLM Engine...")
llm_test = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    gpu_memory_utilization=0.5,
    trust_remote_code=True
)

# Giving slight token room to close the JSON properly, no guided decoding
sampling_params = SamplingParams(temperature=0.0, max_tokens=16)

print("Starting inference...")
prompts = [build_prompt(row) for row in test_rows]

outputs = llm_test.generate(prompts, sampling_params)
generated_texts = [o.outputs[0].text for o in outputs]

buffer = []
for row, raw_text in zip(test_rows, generated_texts):
    parsed = parse_boolean_response(raw_text)
    result = {**row, **parsed}
    buffer.append(result)

df_test_out = pd.DataFrame(buffer)
df_test_out.to_csv(TEST_OUTPUT_PATH, index=False)

print(f"\nSaved 100 test results to {TEST_OUTPUT_PATH}\n")

from vllm.distributed.parallel_state import destroy_model_parallel, destroy_distributed_environment
import torch
import gc

del llm_test
gc.collect()
torch.cuda.empty_cache()
destroy_model_parallel()
destroy_distributed_environment()

pd.set_option('display.max_columns', None)
df_test_out[['instance_id', 'claim_truth', 'eval_is_hallucination', 'eval_raw_output']].head(10)

Preparing 100 rows for Test Run...
Initializing vLLM Engine...
INFO 04-02 16:57:40 [utils.py:261] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'gpu_memory_utilization': 0.5, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-8B'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 04-02 16:57:41 [model.py:541] Resolved architecture: Qwen3ForCausalLM
INFO 04-02 16:57:41 [model.py:1561] Using max model len 40960
INFO 04-02 16:57:41 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-02 16:57:41 [vllm.py:624] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=2957797) INFO 04-02 16:57:44 [core.py:96] Initializing a V1 LLM engine (v0.15.0) with config: model='Qwen/Qwen3-8B', speculative_config=None, tokenizer='Qwen/Qwen3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, di

Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]


(EngineCore_DP0 pid=2957797) INFO 04-02 16:57:54 [default_loader.py:291] Loading weights took 3.59 seconds
(EngineCore_DP0 pid=2957797) INFO 04-02 16:57:54 [gpu_model_runner.py:4118] Model loading took 15.27 GiB memory and 5.964868 seconds
(EngineCore_DP0 pid=2957797) INFO 04-02 16:58:03 [backends.py:805] Using cache directory: /home/aryan_khurana_2026/.cache/vllm/torch_compile_cache/0c2715e170/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=2957797) INFO 04-02 16:58:03 [backends.py:865] Dynamo bytecode transform time: 7.70 s
(EngineCore_DP0 pid=2957797) INFO 04-02 16:58:09 [backends.py:267] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.122 s
(EngineCore_DP0 pid=2957797) INFO 04-02 16:58:09 [monitor.py:34] torch.compile takes 8.82 s in total
(EngineCore_DP0 pid=2957797) INFO 04-02 16:58:10 [gpu_worker.py:356] Available KV cache memory: 6.99 GiB
(EngineCore_DP0 pid=2957797) INFO 04-02 16:58:10 [kv_cache_utils.py:1307] GPU KV cache 

...

(EngineCore_DP0 pid=2957797) INFO 04-02 16:58:16 [gpu_model_runner.py:5051] Graph capturing finished in 6 secs, took 0.62 GiB
(EngineCore_DP0 pid=2957797) INFO 04-02 16:58:16 [core.py:272] init engine (profile, create kv cache, warmup model) took 21.60 seconds
INFO 04-02 16:58:19 [llm.py:343] Supported tasks: ['generate']
Starting inference...


Adding requests:   0%|          | 0/100 [00:00<?, ?it/s]

Processed prompts:   0%|                    | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.…


Saved 100 test results to TEST_RUN_sorted_eval_Llama-3.2-3B-Instruct_1_final.csv



,instance_id,claim_truth,eval_is_hallucination,eval_raw_output
0,casehold_17385_T,True,False,"0\n}\n</think>\n\n{\n ""is_hallucination"": 0"
1,sciq_6328_T,True,None,"2\n}\n</think>\n\n{\n ""is_hallucination"": 2"
2,medmcqa_68979_T,True,False,"0,\n ""explanation"": ""The model disagreed with..."
3,sciq_8677_F,False,False,"0\n}\n</think>\n\n{\n ""is_hallucination"": 0"
4,medmcqa_123940_F,False,True,"1\n}\n</think>\n\n{\n ""is_hallucination"": 1"
5,casehold_33121_F,False,None,"2\n}\n</think>\n\n{\n ""is_hallucination"": 2"
6,sciq_6191_T,True,False,"0\n}\n</think>\n\n{\n ""is_hallucination"": 0"
7,fever_33112,False,True,"1\n}\n</think>\n\n{\n ""is_hallucination"": 1"
8,sciq_191_T,True,False,"0\n}\n</think>\n\n{\n ""is_hallucination"": 0"
9,fever_145303,True,True,"1\n}\n</think>\n\n{\n ""is_hallucination"": 1"


In [3]:
# =========================================
# 4. PRODUCTION RUN (FULL DATASET, CHECKPOINTING)
# =========================================

FINAL_OUTPUT_PATH = "final_eval_" + SORTED_CSV_PATH

print(f"Loading explicitly sorted dataset: {SORTED_CSV_PATH}")
df_prod = pd.read_csv(SORTED_CSV_PATH, low_memory=False)
# Ensure data is absolutely sorted (Shortest to Longest)
df_prod = df_prod.sort_values("prompt_token_length", ascending=True).reset_index(drop=True)

# --- COMPOSITE ROW KEY: instance_id + condition + response ---
# instance_id alone is NOT unique (3 rows per claim: different conditions/responses).
# We use a composite key so each distinct row is tracked independently.
def make_row_key(df):
    cond_str = df['condition'].astype(str) if 'condition' in df.columns else df.get('source_label', '').astype(str)
    return (
        df['instance_id'].astype(str) + '||' +
        cond_str + '||' +
        df['response'].astype(str)
    )

df_prod['_row_key'] = make_row_key(df_prod)

processed_keys = set()
if os.path.exists(FINAL_OUTPUT_PATH):
    print("Loading checkpoint...")
    existing = pd.read_csv(FINAL_OUTPUT_PATH, low_memory=False)
    existing['_row_key'] = make_row_key(existing)
    processed_keys = set(existing['_row_key'])
    print(f"Already processed: {len(processed_keys)} / {len(df_prod)} rows ({100*len(processed_keys)/len(df_prod):.1f}%)")

# Filter out what's already done using the composite key
df_remaining = df_prod[~df_prod['_row_key'].isin(processed_keys)].drop(columns=['_row_key'])
rows = df_remaining.to_dict(orient="records")
print(f"Remaining rows to process: {len(rows)}")

if len(rows) > 0:
    print("Loading Production Model...")
    llm = LLM(
        model=MODEL_ID,
        dtype="bfloat16",
        gpu_memory_utilization=GPU_MEMORY_UTIL,
        max_model_len=3000,  # High Concurrency Limit!
        trust_remote_code=True,
        disable_log_stats=True
    )
    sampling_params = SamplingParams(temperature=0.0, max_tokens=16)

    print("\nExecuting Full Pipeline (Shortest inputs first)...")
    buffer = []

    for i in tqdm(range(0, len(rows), BATCH_SIZE)):
        batch = rows[i:i+BATCH_SIZE]
        prompts = [build_prompt(r) for r in batch]

        try:
            outputs = llm.generate(prompts, sampling_params)
            generated_texts = [o.outputs[0].text for o in outputs]
        except Exception as e:
            print("Batch failed:", str(e))
            continue

        for row, raw_text in zip(batch, generated_texts):
            parsed = parse_boolean_response(raw_text)
            result = {**row, **parsed}
            buffer.append(result)

        chunk_df = pd.DataFrame(buffer)
        if not os.path.exists(FINAL_OUTPUT_PATH):
            chunk_df.to_csv(FINAL_OUTPUT_PATH, index=False)
        else:
            chunk_df.to_csv(FINAL_OUTPUT_PATH, mode="a", header=False, index=False)
        buffer = []

    print(f"\nFULL EVALUATION COMPLETE! Saved to {FINAL_OUTPUT_PATH}")
else:
    print("Dataset already completely processed.")


Loading explicitly sorted dataset: sorted_eval_Llama-3.2-3B-Instruct_1_final.csv
Loading checkpoint...
Already processed: 327003 / 330804 rows (98.9%)
Remaining rows to process: 3299
Loading Production Model...
INFO 04-09 18:04:32 [utils.py:261] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 3000, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-8B'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 04-09 18:04:34 [model.py:541] Resolved architecture: Qwen3ForCausalLM
INFO 04-09 18:04:34 [model.py:1561] Using max model len 3000
INFO 04-09 18:04:34 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=1707229) INFO 04-09 18:04:34 [core.py:96] Initializing a V1 LLM engine (v0.15.0) with config: model='Qwen/Qwen3-8B', speculative_config=None, tokenizer='Qwen/Qwen3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=3000, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoni

Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]


(EngineCore_DP0 pid=1707229) INFO 04-09 18:04:44 [default_loader.py:291] Loading weights took 3.39 seconds
(EngineCore_DP0 pid=1707229) INFO 04-09 18:04:45 [gpu_model_runner.py:4118] Model loading took 15.27 GiB memory and 5.719502 seconds
(EngineCore_DP0 pid=1707229) INFO 04-09 18:04:53 [backends.py:805] Using cache directory: /home/aryan_khurana_2026/.cache/vllm/torch_compile_cache/cdab986631/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=1707229) INFO 04-09 18:04:53 [backends.py:865] Dynamo bytecode transform time: 7.51 s
(EngineCore_DP0 pid=1707229) INFO 04-09 18:04:58 [backends.py:302] Cache the graph of compile range (1, 8192) for later use
(EngineCore_DP0 pid=1707229) INFO 04-09 18:05:01 [backends.py:319] Compiling a graph for compile range (1, 8192) takes 3.65 s
(EngineCore_DP0 pid=1707229) INFO 04-09 18:05:01 [monitor.py:34] torch.compile takes 11.16 s in total
(EngineCore_DP0 pid=1707229) INFO 04-09 18:05:03 [gpu_worker.py:356] Available KV cache memory: 25.99

...

Adding requests:   0%|          | 0/3299 [00:00<?, ?it/s]

Processed prompts:   0%|                   | 0/3299 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.…

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [06:36<00:00, 396.32s/it]


FULL EVALUATION COMPLETE! Saved to final_eval_sorted_eval_Llama-3.2-3B-Instruct_1_final.csv


ERROR 04-09 18:12:28 [core_client.py:605] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.
